## Single Node LLM Pipeline Using Tiny Shakespeare

In this colab, we implement and train a single-node small LLM model on the **Tiny Shakespeare** dataset.

##**`PART 1: Data Loader & Tokenizer - Prep the input`**

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

# 1. Load the tiny shakespeare dataset (~40k lines of text)
ds = tfds.load('tiny_shakespeare', split='train', as_supervised=False)
for ex in ds.take(1): # Transform the text from tensor shape back into text string so we can manipulate it in python
    raw_text = ex['text'].numpy().decode('utf-8')

sample_data_filtered = [block for block in raw_text[:250000].split('\n\n') if block.strip()] # Filter out double newlines / empty lines
T = 100 # Set sequence length value
target_vocab_size = 8000 # Set token vocab size

# 2. Create BPE tokenizer and token vocab
print("Building BPE tokenizer (this takes a few seconds)...")
bpe_tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus( # Tensorflow Dataset Tool tokenizer algorithm
    sample_data_filtered, target_vocab_size=target_vocab_size
)
actual_vocab_size = bpe_tokenizer.vocab_size # Check the final 'actual' vocab size after tokenizer training
print(f"True token vocab size: {actual_vocab_size}")

# 3. Create training sequences
encoded_list = [bpe_tokenizer.encode(s) for s in sample_data_filtered] # Tokenize the data set

encoded_padded = tf.keras.preprocessing.sequence.pad_sequences( # Create uniform length sequences for training (aka T!)
    encoded_list, maxlen=T + 1, padding='post', truncating='post' # Add 0s if the chunk of text is shorter than T, truncate the text if longer than T (not ideal but we'll do this for now)
)

training_dataset = encoded_padded[:, :-1] # Create training data set, minus the last one
target_dataset = encoded_padded[:, 1:] # Create target data set - just shifted over by 1

print(f"Training Data Shape: {training_dataset.shape}")

Building BPE tokenizer (this takes a few seconds)...
True token vocab size: 7749
Training Data Shape: (1771, 100)


###**Helper Section: Visualize the BPE tokenizer**

In [ ]:
import numpy as np

# The vocabulary is stored inside the tokenizer object as a list!
vocab = bpe_tokenizer.subwords # Subwords is where the vocab is auto stored when we created the tokenizer

def decode_opt_sequence(indices):
    valid_indices = [int(i) for i in indices if i != 0] # Filter out the 0 padding, to decode just the text
    return bpe_tokenizer.decode(valid_indices) # Call the decoder to decode tokens back to text

print("--- TOKENIZED TEXT VISUALIZER ---")
for i in range(3):
    print(f"\nSample #{i+1}:")
    print(f"Context (X) Text:  '{decode_opt_sequence(training_dataset[i])}'")
    print(f"Tokenized IDs (X): {training_dataset[i][:20]}...")

--- TOKENIZED TEXT VISUALIZATION ---

Sample #1:
Context (X) Text:  'First Citizen:
Before we proceed any further, hear me speak.'
Tokenized IDs (X): [  53   81    2  677   39 1355  217  599    1  149   37  204 7539    0
    0    0    0    0    0    0]...

Sample #2:
Context (X) Text:  'All:
Speak, speak.'
Tokenized IDs (X): [ 430    2 1544    1  204 7539    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]...

Sample #3:
Context (X) Text:  'First Citizen:
You are all resolved rather to die than to famish?'
Tokenized IDs (X): [  53   81    2   69   40   47 1714  436    6  564   88    6 2747 7556
    0    0    0    0    0    0]...


##**PART 2: Model & Training Loop**

In [ ]:
d_model = 256 # Establish embedding vector size (E) - this represents how many numbers each token will be represented by
num_layers = 12 # Establish number of layers in our model

input = tf.keras.Input(shape=(T,)) # Create a tensor shape object that will be the input to the model

# 1. Create Positional Embedding layer - this layer adds context to tokens about their position within their token sequence (T)
class PositionalEmbedding(keras.layers.Layer):
    def __init__(self, vocab_size, d_model, seq_len, **kwargs):
        super().__init__(**kwargs)
        self.word_embeddings = Embedding(vocab_size, d_model, mask_zero=True) # This is the core Embedding dictionary
        self.position_embeddings = Embedding(seq_len, d_model) # This is the Positional Embedding dictionary; row 0 represents 1st token position, row 1 represents 2nd token position, etc.

    # Incorporate the positional info into the token's core info base
    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1) # Creates a list of indices representing token position [0,1,2, etc.]
        embedded_words = self.word_embeddings(inputs) # Looks up 256 embedding vector of each token in the input
        embedded_pos = self.position_embeddings(positions) # Looks up the positional embedding vector for each position from Positional Embedding dictionary we created earlier
        return embedded_words + embedded_pos # Add the positional encoding info to each token's embedding vector

x = PositionalEmbedding(actual_vocab_size, d_model, T)(input) # Instantiate the embedding vector with positional info baked in

# 2. Create Attention and FFN layers

# ATTN layer - this layer adds in context of surrounding tokens
def attention_block(x, num_heads, key_dim, dropout_rate=0.1): # key_dim represents how much info each head processes
    # Use causal masking (use_causal_mask=True) to prevent looking at tokens that come after - that would allow the model to "cheat"
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x, use_causal_mask=True)
    attn_output = Dropout(dropout_rate)(attn_output) # This randomly turns off a portion of the neurons to prevent overfitting - this technique is called 'Droput'
    x = Add()([x, attn_output]) # Add the attention output to the tokens - now our tokens have context
    x = LayerNormalization()(x) # Normalize the combined data to keep the scale of the math stable through layers
    return x

# FFN layer - this layers adds in token info / knowledge
def ffn_block(x, ffn_dim, dropout_rate=0.1):
    ffn_output = Dense(ffn_dim, activation='relu')(x) # Expand the shape of each token to ffn_dim, and apply relu activation to incorporate token 'feature' learnings
    ffn_output = Dense(x.shape[-1])(ffn_output) # Compress the tokens back down to the size of d_model
    ffn_output = Dropout(dropout_rate)(ffn_output) # Randomly zero out some of the learnings - similar to in attn
    x = Add()([x, ffn_output]) # Add the ffn output to the tokens - now our tokens have knowledge and info about what they represent
    x = LayerNormalization()(x) # Normalize again for same reasons discussed earlier
    return x

# 3. Create the deep layer transformer by stacking the ATTN and FFN layers - using num_layers we set earlier (12)
num_heads = 8
for _ in range(num_layers):
    x = attention_block(x, num_heads=num_heads, key_dim=d_model // num_heads)
    x = ffn_block(x, ffn_dim=2048) # ffn_dim is typically significant larger than token embedding size (d_model); why? you're letting it learn in a larger space to understand more complex relationships

outputs = tf.keras.layers.Dense(actual_vocab_size, activation='softmax')(x) # Project the output into the full space of the vocab, so you can get a probability for each possible next token. The token with the highest probability is your next token!

# 4. Learning Rate Scheduler
# This layer adjust the learning rate as you go, start by taking bigger steps and gradually take finer steps - you optimize for learning fast, and then get more precise
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=3e-4,
    decay_steps=2000, # Decrease the learning rate every 2000 batches
    decay_rate=0.9) # This sets how much to shrink the LR by
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

# 5. Compile the deep layer model
model = tf.keras.Model(inputs=input, outputs=outputs) # Package all 12 layers up into a single model object
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy')

print(f"Tiny Shakespeare model is Ready: {num_layers} layers.")

Tiny Shakespeare model is Ready: 12 layers.


##**PART 3: Training Looper** - Run this cell multiple times to keep training without losing progress!

In [ ]:
print("Training Tiny Shakespeare model (12 layers)...")
# We'll use a slightly larger batch size to stabilize gradients for the deeper network
model.fit(training_dataset, target_dataset, epochs=100, batch_size=128, verbose=1)

Training Tiny Shakespeare model (12 layers)...
Epoch 1/100
14/14 [==============================] - 29s 256ms/step - loss: 5.9663
Epoch 2/100
14/14 [==============================] - 3s 243ms/step - loss: 4.2586
Epoch 3/100
14/14 [==============================] - 3s 241ms/step - loss: 3.1007
Epoch 4/100
14/14 [==============================] - 3s 247ms/step - loss: 2.5886
Epoch 5/100
14/14 [==============================] - 3s 243ms/step - loss: 2.3868
Epoch 6/100
14/14 [==============================] - 3s 241ms/step - loss: 2.1272
Epoch 7/100
14/14 [==============================] - 3s 230ms/step - loss: 2.0259
Epoch 8/100
14/14 [==============================] - 3s 246ms/step - loss: 1.9733
Epoch 9/100
14/14 [==============================] - 3s 234ms/step - loss: 1.9411
Epoch 10/100
14/14 [==============================] - 3s 228ms/step - loss: 1.9101
Epoch 11/100
14/14 [==============================] - 3s 222ms/step - loss: 1.8997
Epoch 12/100
14/14 [============================

**PART 4: TEXT GENERATOR (INFERENCE)**

In [ ]:
import numpy as np
import tensorflow as tf

def generate_shakespeare(seed_text, gen_length=150, temperature=0.7):
    """Generates text using the 12-layer model and true BPE/Subword tokenizer."""
    result_text = seed_text

    print(f"--- Generating with TRUE Sub-word Model (T={T_optimized}) ---")

    for i in range(gen_length):
        # 1. Encode text to tokens
        input_tokens = bpe_tokenizer.encode(result_text)

        # 2. Pad or truncate to context window, keeping the END of the sequence
        input_padded = tf.keras.preprocessing.sequence.pad_sequences(
            [input_tokens], maxlen=T, padding='pre', truncating='pre'
        )

        # 3. Predict
        preds = model.predict(input_padded, verbose=0)[0, -1, :]

        # Masking padding (0)
        preds[0] = 0.0

        # 4. Sample with Temperature
        preds = preds / (np.sum(preds) + 1e-8)
        logits = np.log(preds + 1e-8) / temperature
        next_token_id = tf.random.categorical([logits], num_samples=1)[0, 0].numpy()

        # 5. Decode and append to raw string
        # The true SubwordTextEncoder handles spaces properly upon decode
        next_word = bpe_tokenizer.decode([next_token_id])
        result_text += next_word

    return result_text

# --- TEST THE TRUE BPE INFERENCE ---
prompt = "First Citizen:"
print(generate_shakespeare(prompt, gen_length=100, temperature=0.6))

--- Generating with TRUE Sub-word Model (T=100) ---
First Citizen:, good and , , , ,
', , ,
,
your Your , 

, to his ', their Sirrahcolour; , I , , ', and , , a with they ; shall , , , ., And not , I have ,
the and , when have the not , and -my , most , , .my , , ?such , ', , the of he , 
, and , , the ''a , I with the have , , ., , , 


**Evaluation Metric: Perplexity** - Perplexity is $e^{loss}$. It represents the geometric mean of the number of tokens the model is choosing from at each step. A perplexity of 10 means the model is as confused as if it had to choose between 10 equally likely words. This is one score used to understand the quality of the model by judging its coherency.

In [ ]:
import math

def evaluate_model(model, dataset_X, dataset_Y):
    loss = model.evaluate(dataset_X, dataset_Y, verbose=0)
    perplexity = math.exp(loss)
    print(f"Final Cross-Entropy Loss: {loss:.4f}")
    print(f"Model Perplexity:        {perplexity:.2f}")
    return perplexity

# Dynamic evaluation based on what is available
if 'optimized_model' in globals():
    print("### Grading Optimized Model (Wordpiece/Large) ###")
    evaluate_model(optimized_model, X_opt, Y_opt)
elif 'shakespeare_model_v2' in globals():
    print("### Grading Original Model (Keras Vectorizer) ###")
    evaluate_model(shakespeare_model_v2, X_shakespeare, Y_shakespeare)
else:
    print("No models found. Please run a training cell first (e9671b0a or 8d877f62).")

### Grading Optimized Model (Wordpiece/Large) ###
Final Cross-Entropy Loss: 8.8169
Model Perplexity:        6747.15
